## Qwen3-VL-8B LoRA Test 추론 및 submission 생성

선택한 LoRA adapter로 Test 819개를 추론하고 `submission.csv`를 저장합니다. Test 데이터의 분포나 특징은 분석하지 않습니다. 이후 다른 모델을 사용할 때는 1번 셀의 실험 이름과 adapter 경로만 바꾸면 됩니다.

### 1. 경로와 추론 설정

기본값은 현재 Validation Exact Match가 가장 높은 `fold0_aug1/epoch1_adapter`입니다. A100에서는 batch 4를 사용하고, 메모리가 부족하거나 RTX 3090에서 실행할 때는 batch 1로 변경합니다.

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임이 아닙니다.'
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/SNU')
DATA_DIR = PROJECT_DIR / 'data'
TEST_CSV = DATA_DIR / 'test.csv'
TEST_IMAGE_DIR = DATA_DIR / 'test'
SAMPLE_SUBMISSION = DATA_DIR / 'sample_submission.csv'

MODEL_NAME = 'Qwen/Qwen3-VL-8B-Instruct'
EXPERIMENT_NAME = 'fold0_aug1_epoch1'
ADAPTER_DIR = (
    PROJECT_DIR / 'outputs' / 'qwen3_vl_8b_lora'
    / 'fold0_aug1' / 'epoch1_adapter'
)
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'submissions' / EXPERIMENT_NAME
SUBMISSION_PATH = OUTPUT_DIR / 'submission.csv'

INFERENCE_BATCH_SIZE = 4  # RTX 3090에서는 1
MAX_NEW_TOKENS = 16

for path in (TEST_CSV, TEST_IMAGE_DIR, SAMPLE_SUBMISSION, ADAPTER_DIR):
    assert path.exists(), f'경로가 없습니다: {path}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Adapter: {ADAPTER_DIR}')
print(f'Submission: {SUBMISSION_PATH}')

### 2. 필요한 패키지 설치

학습 노트북과 동일한 Transformers·PEFT·Qwen 이미지 처리 버전을 설치합니다.

In [ ]:
%pip install -q transformers==5.13.0 accelerate==1.14.0 peft==0.19.1 qwen-vl-utils==0.0.14
%pip uninstall -y torchao

### 3. Test 로드와 학습 프롬프트 정의

Test와 sample submission의 819개 ID 순서가 같은지만 확인합니다. 프롬프트와 chronological order 변환은 학습 및 Validation에서 사용한 것과 동일합니다.

In [ ]:
import ast
import re

import pandas as pd
from tqdm.auto import tqdm

test = pd.read_csv(TEST_CSV, dtype=str, encoding='utf-8-sig')
sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION, dtype=str, encoding='utf-8-sig'
)
assert test.columns.tolist() == [
    'Id', 'Input_1', 'Input_2', 'Input_3', 'Input_4', 'Sentence'
]
assert sample_submission.columns.tolist() == ['Id', 'Answer']
assert len(test) == len(sample_submission) == 819
assert test['Id'].tolist() == sample_submission['Id'].tolist()

IDENTITY_ORDER = (1, 2, 3, 4)
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE,
)
STRUCTURED_HINT = (
    'This caption describes distinct sequential stages, often marked by words like '
    "'then', 'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    'This caption describes a single continuous action or a brief moment. '
    'Focus on subtle visual cues such as camera movement direction, object position changes, '
    'hand/body motion, and lighting changes to infer the temporal order.'
)

def split_clauses(sentence, max_clauses=4):
    sentence = sentence.strip()
    parts = CLAUSE_PATTERN.split(sentence)
    clauses = []
    current = parts[0].strip(' ,;.')
    for index in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[index + 1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [clause for clause in clauses if clause]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses - 1] + [' '.join(clauses[max_clauses - 1:])]
    return clauses or [sentence]

def chronological_order_to_answer(order):
    answer = [0, 0, 0, 0]
    for position, image_number in enumerate(order, start=1):
        answer[image_number - 1] = position
    return answer

def build_prompt_messages(row):
    sentence = row['Sentence']
    hint = STRUCTURED_HINT if len(split_clauses(sentence)) >= 3 else CONTINUOUS_HINT
    content = []
    for image_number in IDENTITY_ORDER:
        image_path = TEST_IMAGE_DIR / row['Id'] / row[f'Input_{image_number}']
        content.append({'type': 'image', 'image': str(image_path)})
        content.append({'type': 'text', 'text': f'\nImage {image_number}\n'})
    content.append({
        'type': 'text',
        'text': (
            f'Thinking about the sentence: "{sentence}"\n'
            f'{hint}\n'
            'Look at the 4 images above labeled Image 1 to Image 4. '
            'Determine the correct chronological order of these images to match the sentence. '
            'Provide the answer ONLY as a Python list of integers. '
            'Example: [1, 2, 3, 4]'
        ),
    })
    return [{'role': 'user', 'content': content}]

def parse_output(text):
    start = text.find('[')
    end = text.rfind(']')
    if start == -1 or end == -1 or start >= end:
        return None
    try:
        order = ast.literal_eval(text[start:end + 1])
    except (SyntaxError, ValueError):
        return None
    if not isinstance(order, list) or len(order) != 4 or sorted(order) != [1, 2, 3, 4]:
        return None
    return chronological_order_to_answer([int(number) for number in order])

print(f'Test samples: {len(test):,}')

### 4. 모델과 LoRA adapter 로드

Qwen3-VL-8B 백본과 선택한 LoRA adapter 하나를 추론 모드로 불러옵니다.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel
from qwen_vl_utils import process_vision_info

processor = AutoProcessor.from_pretrained(MODEL_NAME)
processor.tokenizer.padding_side = 'left'

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map='auto',
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)
model.config.use_cache = True
model.eval()

print('모델과 adapter 로드 완료')

### 5. Test 추론 및 submission.csv 저장

Test 819개를 순서대로 추론합니다. 모든 출력이 네 숫자의 순열로 변환되면 sample submission의 ID 순서를 유지해 `submission.csv`를 저장합니다.

In [ ]:
predictions = []
parse_failure_count = 0

for start_index in tqdm(
    range(0, len(test), INFERENCE_BATCH_SIZE),
    desc='Qwen3-VL Test inference',
):
    batch_rows = test.iloc[start_index:start_index + INFERENCE_BATCH_SIZE]
    batch_messages = [
        build_prompt_messages(row)
        for _, row in batch_rows.iterrows()
    ]
    batch_texts = [
        processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        for messages in batch_messages
    ]
    all_images = []
    for messages in batch_messages:
        image_inputs, video_inputs = process_vision_info(messages)
        assert not video_inputs
        all_images.extend(image_inputs)
    inputs = processor(
        text=batch_texts,
        images=all_images,
        padding=True,
        return_tensors='pt',
    ).to(model.device)

    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            num_beams=1,
        )
    generated_only = generated[:, inputs['input_ids'].shape[1]:]
    output_texts = processor.batch_decode(
        generated_only,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    for raw_output in output_texts:
        answer = parse_output(raw_output)
        if answer is None:
            parse_failure_count += 1
        predictions.append(answer)

assert len(predictions) == len(test) == 819
assert parse_failure_count == 0, f'파싱 실패: {parse_failure_count}건'

submission = sample_submission[['Id']].copy()
submission['Answer'] = [str(answer) for answer in predictions]
submission.to_csv(SUBMISSION_PATH, index=False, encoding='utf-8-sig')

saved_submission = pd.read_csv(
    SUBMISSION_PATH, dtype=str, encoding='utf-8-sig'
)
assert saved_submission.columns.tolist() == ['Id', 'Answer']
assert len(saved_submission) == 819
assert saved_submission['Id'].tolist() == test['Id'].tolist()

print(f'Submission 저장 완료: {SUBMISSION_PATH}')

### 6. submission.csv 내려받기

5번 셀이 완료된 후 실행합니다.

In [ ]:
from google.colab import files
files.download(str(SUBMISSION_PATH))